In [2]:
# ==============================================================================
# NANDI PRECISION AGRICULTURE: STANDALONE RECOMMENDER SYSTEM (OFFLINE VERSION)
# ==============================================================================
import os
import json
import numpy as np
import pandas as pd
import rasterio
from google.colab import drive

# --- 1. MOUNT GOOGLE DRIVE ---
drive.mount('/content/drive')

# --- 2. DEFINE SYSTEM PATHS ---
BASE = '/content/drive/MyDrive/02_NandiSeedRecommender2'
SEED_XLSX_PATH = os.path.join(BASE, 'Seed_Data/KenyaSeedWebScrape.xlsx')

# --- 3. LOAD EXCEL SEED DATABASE ---
if os.path.exists(SEED_XLSX_PATH):
    seed_df = pd.read_excel(SEED_XLSX_PATH, engine='openpyxl')
    seed_df.columns = seed_df.columns.str.strip()
else:
    seed_df = pd.DataFrame()

# --- 4. SEED RANKING LOGIC ---
def rank_seed_varieties(seed_df, elevation, precip, stress_types):
    if seed_df.empty: return pd.DataFrame(columns=['Variety', 'Note'])
    df = seed_df.copy()
    def check_requirements(row):
        issues = []
        try:
            if not (row["Elevation Min"] <= elevation <= row["Elevation Max"]):
                issues.append("Elevation")
            if not (row["Precipitation Min"] <= precip <= row["Precipitation Max"]):
                issues.append("Rainfall")
            if "dry" in stress_types and row.get("Moisture-Stress Tolerant") != 1:
                issues.append("Not drought-tolerant")
            if "heat" in stress_types and row.get("Heat Tolerant") != 1:
                issues.append("Not heat-tolerant")
            if "cold" in stress_types and row.get("Cold Tolerant") != 1:
                issues.append("Not cold-tolerant")
        except: return "Data formatting error"
        return ", ".join(issues) if issues else "Meets all requirements"

    df["RequirementNotes"] = df.apply(check_requirements, axis=1)
    df["YieldScore"] = pd.to_numeric(df["Potential yield (t/Ha)"], errors="coerce")
    df['PerfectMatch'] = df['RequirementNotes'] == "Meets all requirements"
    df = df.sort_values(["PerfectMatch", "YieldScore"], ascending=[False, False])
    return df.head(5)

# --- 5. THE MASTER REPORT FUNCTION ---
def get_comprehensive_nandi_report(lat, lon, seed_df, season='LongRains'):
    FINAL_DIR = os.path.join(BASE, 'Final_Outputs')
    FACTORS_DIR = os.path.join(FINAL_DIR, f'Factors_{season}')
    RAW_DIR = os.path.join(FINAL_DIR, f'Raw_Values_{season}')

    # Units for display
    UNITS = {
        'ph': '', 'total_nitrogen': '%', 'phosphorus': 'mg/kg', 'potassium': 'cmol/kg',
        'calcium': 'cmol/kg', 'magnesium': 'cmol/kg', 'organic_carbon': '%',
        'ecec': 'cmol/kg', 'zinc': 'mg/kg', 'iron': 'mg/kg', 'clay_content': '%',
        'slope': '%', 'rain': 'mm', 'temp': '°C', 'rh': '%', 'bedrock_depth': 'cm',
        'elevation': 'm', 'stone_content': '%', 'cec_apparent': 'cmol/kg', 'texture': '',
        'min_temp': '°C', 'max_temp': '°C', 'germin_temp': '°C',
        'rh_dev': '%', 'rh_mat': '%',
        'prec_month1': 'mm', 'prec_month2': 'mm', 'prec_month3': 'mm', 'prec_month4': 'mm'
    }

    # iSDA texture class integer → name                                      # ADD
    TEXTURE_NAMES = {                                                         # ADD
        1:  'Clay',                                                           # ADD
        2:  'Silty Clay',                                                     # ADD
        3:  'Silty Clay Loam',                                                # ADD
        4:  'Sandy Clay',                                                     # ADD
        5:  'Sandy Clay Loam',                                                # ADD
        6:  'Clay Loam',                                                      # ADD
        7:  'Silt',                                                           # ADD
        8:  'Silt Loam',                                                      # ADD
        9:  'Loam',                                                           # ADD
        10: 'Sandy Loam',                                                     # ADD
        11: 'Loamy Sand',                                                     # ADD
        12: 'Sand',                                                           # ADD
    }                                                                         # ADD

    # Load Context
    json_path = os.path.join(FINAL_DIR, 'County_Averages.json')
    county_ref = {'scores': {}, 'raw': {}}
    if os.path.exists(json_path):
        with open(json_path, 'r') as f:
            county_ref = json.load(f).get(season, {})

    # Extract Suitability
    suit_path = os.path.join(FINAL_DIR, f'Suit_Mean_{season}.tif')
    with rasterio.open(suit_path) as src:
        row, col = src.index(lon, lat)
        suit_mu = src.read(1)[row, col]
        if np.isnan(suit_mu): return "Coordinates outside Nandi mask."
    with rasterio.open(os.path.join(FINAL_DIR, f'Suit_Std_{season}.tif')) as src:
        suit_std = src.read(1)[row, col]

    def get_raw_val(var):
        lookup = var
        if var == 'rain': lookup = 'mean_season_rain'
        if var == 'temp': lookup = 'mean_temp'
        if var == 'rh':   lookup = 'rh_dev'
        path = os.path.join(RAW_DIR, f'{lookup}_raw.tif')
        if not os.path.exists(path) and var == 'elevation':
            path = os.path.join(FINAL_DIR, 'Nandi_Elevation_30m.tif')
        if os.path.exists(path):
            with rasterio.open(path) as s:
                r, c = s.index(lon, lat)
                return s.read(1)[r, c]
        return np.nan

    elev_val   = get_raw_val('elevation')
    precip_val = get_raw_val('rain')
    temp_val   = get_raw_val('temp')

    stress_codes = []
    if not np.isnan(temp_val):           # ADD: guard against NaN temp
        if temp_val < 16: stress_codes.append("cold")
        if temp_val > 30: stress_codes.append("heat")
    if not np.isnan(precip_val):         # ADD: guard against NaN precip
        if precip_val < 450: stress_codes.append("dry")

    recommendations = rank_seed_varieties(seed_df, elevation=elev_val, precip=precip_val, stress_types=stress_codes)

    # Define Section Variables
    soil_vars = {'ph', 'total_nitrogen', 'phosphorus', 'potassium', 'calcium', 'magnesium',
                 'organic_carbon', 'ecec', 'cec_apparent', 'zinc', 'iron', 'clay_content'}

    climate_vars = {'rain', 'temp', 'rh', 'min_temp', 'max_temp', 'germin_temp',
                    'rh_dev', 'rh_mat', 'prec_month1', 'prec_month2', 'prec_month3', 'prec_month4'}

    # Mirrors get_raw_val remapping for county avg JSON lookup
    raw_key_map = {
        'rain': 'mean_season_rain',
        'temp': 'mean_temp',
        'rh':   'rh_dev',
    }

    soil_list, climate_list, phys_list = [], [], []

    factor_files = sorted([f for f in os.listdir(FACTORS_DIR) if f.endswith('_mean.tif')])

    for f_name in factor_files:
        var = f_name.replace('_mean.tif', '')

        if var.startswith('prob_'): continue  # already shown in risks section

        with rasterio.open(os.path.join(FACTORS_DIR, f_name)) as m_f, \
             rasterio.open(os.path.join(FACTORS_DIR, f'{var}_std.tif')) as s_f:
            r, c = m_f.index(lon, lat)
            score, unc = m_f.read(1)[r, c], s_f.read(1)[r, c]

        if var == 'texture':                                                  # ADD: texture special case
            raw_cls = get_raw_val('texture')                                  # ADD
            if not np.isnan(raw_cls):                                         # ADD
                actual_name = TEXTURE_NAMES.get(int(round(raw_cls)), 'Unknown')  # ADD
            else:                                                             # ADD
                actual_name = 'N/A'                                           # ADD
            avg_cls_raw = county_ref.get('raw', {}).get('texture', np.nan)   # ADD
            if avg_cls_raw and not np.isnan(float(avg_cls_raw)):              # ADD
                avg_name = TEXTURE_NAMES.get(int(round(float(avg_cls_raw))), 'Unknown')  # ADD
            else:                                                             # ADD
                avg_name = 'N/A'                                              # ADD
            entry = {'var': var, 'score': score, 'unc': unc,                 # ADD
                     'actual': actual_name, 'avg_raw': avg_name, 'unit': ''}  # ADD
        else:                                                                 # ADD
            json_key = raw_key_map.get(var, var)
            entry = {'var': var, 'score': score, 'unc': unc, 'actual': get_raw_val(var),
                     'avg_raw': county_ref.get('raw', {}).get(json_key, 0), 'unit': UNITS.get(var, '')}

        if var in soil_vars: soil_list.append(entry)
        elif var in climate_vars: climate_list.append(entry)
        else: phys_list.append(entry)

    soil_list.sort(key=lambda x: x['score'])
    climate_list.sort(key=lambda x: x['score'])
    phys_list.sort(key=lambda x: x['score'])

    # Risks
    risk_labels = ['Cold Risk', 'Heat Risk', 'Drought Risk', 'Flood Risk', 'Overall Failure']
    risk_key_map = {
        'Cold Risk':       'prob_cold',
        'Heat Risk':       'prob_heat',
        'Drought Risk':    'prob_drought',
        'Flood Risk':      'prob_flood',
        'Overall Failure': 'prob_overall_fail'
    }
    risk_data = []
    with rasterio.open(os.path.join(FINAL_DIR, f'Risks_Mean_{season}.tif')) as m_f, \
         rasterio.open(os.path.join(FINAL_DIR, f'Risks_Std_{season}.tif')) as s_f:
        mu_r = m_f.read(window=((row, row+1), (col, col+1)))[:, 0, 0]
        st_r = s_f.read(window=((row, row+1), (col, col+1)))[:, 0, 0]
        for i, label in enumerate(risk_labels):
            risk_data.append({'label': label, 'mu': mu_r[i], 'std': st_r[i],
                              'avg': county_ref.get('scores', {}).get(risk_key_map[label], 0)})

    # --- PRINTING ---
    GREY  = '\033[48;5;250m\033[38;5;16m'
    RESET = '\033[0m'
    W = 115
    print(f"\n{'='*W}\n NANDI PRECISION AGRICULTURE: COMPREHENSIVE REPORT | {season}")
    print(f" GPS: {lat}, {lon} | Elevation: {elev_val:.0f}m | Overall Suitability: {suit_mu*100:.1f}% (±{suit_std:.3f})\n{'='*W}")

    print(f"\n{'TOP SEED RECOMMENDATIONS':<50}\n" + "-" * W)
    print(recommendations[['Variety', 'Potential yield (t/Ha)', 'RequirementNotes']].to_string(index=False))

    header = f"{'VARIABLE':<20} | {'MEASURED':<18} | {'COUNTY AVG':<18} | {'SCORE':<8} | {'UNCERTAINTY'}"
    print(f"\n{header}\n" + "-" * W + "\nCLIMATE RISKS (10-Year Probabilities)")
    for r in risk_data:
        print(f"{r['label']:<20} | {r['mu']*100:>17.1f}% | {r['avg']*100:>17.1f}% | {r['mu']:>8.3f} | ±{r['std']:.3f}")

    def print_section(title, data_list):
        if not data_list: return
        print(f"\n{title}")
        for i, s in enumerate(data_list):
            if isinstance(s['actual'], str):                                  # ADD: texture has string values
                act_str = s['actual']                                         # ADD
                avg_str = s['avg_raw'] if isinstance(s['avg_raw'], str) else 'N/A'  # ADD
            else:
                act_str = f"{s['actual']:>8.2f} {s['unit']}".strip() if not np.isnan(s['actual']) else "N/A"
                avg_str = f"{s['avg_raw']:>8.2f} {s['unit']}".strip() if s['avg_raw'] > 0 else "N/A"
            row_str = f"{s['var']:<20} | {act_str:<18} | {avg_str:<18} | {s['score']:>8.3f} | ±{s['unc']:.3f}"
            if i < 3: print(f"{GREY}{row_str}{RESET}")
            else: print(row_str)

    print_section("SOIL NUTRIENTS & PROPERTIES", soil_list)
    TEMP_ORDER  = ['temp', 'min_temp', 'max_temp', 'germin_temp']           # ADD
    RAIN_ORDER  = ['rain', 'prec_month1', 'prec_month2',                    # ADD
                   'prec_month3', 'prec_month4']                            # ADD
    RH_ORDER    = ['rh', 'rh_mat', 'rh_dev']                               # ADD

    def sorted_subgroup(var_list, order):                                   # ADD
        lookup = {e['var']: e for e in var_list}                           # ADD
        return [lookup[v] for v in order if v in lookup]                   # ADD

    climate_map = {e['var']: e for e in climate_list}                      # ADD
    print_section("TEMPERATURE",   sorted_subgroup(climate_list, TEMP_ORDER))   # ADD
    print_section("PRECIPITATION", sorted_subgroup(climate_list, RAIN_ORDER))   # ADD
    print_section("HUMIDITY",      sorted_subgroup(climate_list, RH_ORDER))     # ADD
    print_section("PHYSICAL & TOPOGRAPHY", phys_list)
    print(f"{'='*W}\n")

# --- EXECUTE ---
get_comprehensive_nandi_report(0.2, 35.30, seed_df, season='LongRains')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

 NANDI PRECISION AGRICULTURE: COMPREHENSIVE REPORT | LongRains
 GPS: 0.2, 35.3 | Elevation: 2096m | Overall Suitability: 60.4% (±0.004)

TOP SEED RECOMMENDATIONS                          
-------------------------------------------------------------------------------------------------------------------
Variety  Potential yield (t/Ha) RequirementNotes
  H6218                   12.32         Rainfall
  H6213                   11.88         Rainfall
  H9401                   11.00         Rainfall
  H6210                   11.00         Rainfall
   H629                   10.56         Rainfall

VARIABLE             | MEASURED           | COUNTY AVG         | SCORE    | UNCERTAINTY
-------------------------------------------------------------------------------------------------------------------
CLIMATE RISKS (10-Year Probabilities)
Cold Risk            |       

#Pushing

In [ ]:
import os
import shutil
from google.colab import drive
from google.colab import userdata

# 1. Mount Drive
drive.mount('/content/drive', force_remount=True)

# 2. Configuration
REPO_PATH = "/content/drive/MyDrive/01_Github"
SOURCE_FOLDER = "/content/drive/MyDrive/02_NandiSeedRecommender2"
DESTINATION_PATH = os.path.join(REPO_PATH, "NandiSeedRecommender2")
TOKEN = userdata.get('GITHUB_TOKEN')
REMOTE_URL = f"https://{TOKEN}@github.com/Samarnorld/smartseed-backend.git"

# 3. Helper to skip Google Docs (fixes Errno 95)
def ignore_google_docs(dirname, filenames):
    return [f for f in filenames if f.endswith(('.gdoc', '.gsheet', '.gslides'))]

# 4. Copy Folder into Repo
print("Copying files... this may take a few minutes for 1GB.")
if os.path.exists(SOURCE_FOLDER):
    if os.path.exists(DESTINATION_PATH):
        shutil.rmtree(DESTINATION_PATH)
    shutil.copytree(SOURCE_FOLDER, DESTINATION_PATH, ignore=ignore_google_docs)
    print("Copy complete.")
else:
    print("Source folder not found.")

# 5. Navigate and Configure Git
%cd {REPO_PATH}
!git config --global user.email "harryfyjiswalker@example.com"
!git config --global user.name "harryfyjiswalker"
# Increase buffer for 1GB push
!git config --global http.postBuffer 1048576000

# 6. Check for files > 100MB (GitHub's limit)
print("Checking for large files...")
for root, dirs, files in os.walk(DESTINATION_PATH):
    for f in files:
        fp = os.path.join(root, f)
        if os.path.getsize(fp) > 100 * 1024 * 1024:
            print(f"WARNING: {f} is over 100MB and will cause the push to fail.")

# 7. Push to GitHub
!git add .
!git commit -m "Add NandiSeedRecommender2 folder"
!git pull {REMOTE_URL} main --rebase
!git push {REMOTE_URL} main

Mounted at /content/drive
Copying files... this may take a few minutes for 1GB.


ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.

ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.

ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.



Copy complete.
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py", line 3553, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "/tmp/ipython-input-3220002930.py", line 31, in <cell line: 0>
    get_ipython().run_line_magic('cd', '{REPO_PATH}')
  File "/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py", line 2418, in run_line_magic
    result = fn(*args, **kwargs)
             ^^^^^^^^^^^^^^^^^^^
  File "<decorator-gen-85>", line 2, in cd
  File "/usr/local/lib/python3.12/dist-packages/IPython/core/magic.py", line 187, in <lambda>
    call = lambda f, *a, **k: f(*a, **k)
                              ^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/IPython/core/magics/osm.py", line 342, in cd
    oldcwd = os.getcwd()
             ^^^^^^^^^^^
OSError: [Errno 107] Transport endpoint is not connected

During handling of the above exception, another exception occurred:


In [6]:
!git add '/content/drive/MyDrive/02_NandiSeedRecommender2' .

fatal: not a git repository (or any parent up to mount point /content)
Stopping at filesystem boundary (GIT_DISCOVERY_ACROSS_FILESYSTEM not set).


In [ ]:
!git commit -m "Add NandiSeedRecommender2 folder and new files"

In [ ]:
from google.colab import userdata
TOKEN = userdata.get('GITHUB_TOKEN')

!git pull https://{TOKEN}@github.com/Samarnorld/smartseed-backend.git main --rebase

In [ ]:
!git push https://{TOKEN}@github.com/Samarnorld/smartseed-backend.git main

In [4]:
from google.colab import userdata
import os

USERNAME = "harryfyjiswalker"
REPO_OWNER = "Samarnorld"
REPO_NAME = "smartseed-backend"
TOKEN = userdata.get('GITHUB_TOKEN')